# Track 1 · Step 1: download SeaDronesSee and UAVDT to Drive and explore them

AI251P Capstone, Phase 1 (M1). Owner: Abdulmohsen Alharthi.

What this notebook does:
1. Mounts Google Drive and keeps **one copy of each original archive** in `01_Data/Raw/` on Drive.
2. Extracts the archives to the Colab local disk (`/content/data`), which is much faster than reading from Drive.
3. Explores both datasets: files, splits, classes, box sizes, metadata (flight / video ids for the leakage-safe split), and a quick check for unreadable images.
4. Saves the summary as `01_Data/YYYY-MM-DD_Step1Summary_vN.json`, and the CSV tables and sample images in `01_Data/YYYY-MM-DD_Step1Exploration_vN/` (team naming rule in `00_README.txt`).

Drive layout used: `AI251P_Capstone_CrossDomainAerial/01_Data/Raw/SeaDronesSee` and `.../Raw/UAVDT`. The private Al-Hada folder is never read or written by this notebook.

Rules from the guidelines (Section 2.4): the data are never committed to GitHub and never redistributed. VSAI and the Al-Hada footage are **not** used here.

Runtime: CPU is enough for this step (no GPU needed).

## 0. Settings — edit these first

In [8]:
# ---- edit these ----
# Run Colab with the KAUST account, the owner of the project folder.
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'
DATA_DIR      = f'{DRIVE_PROJECT}/01_Data'

# SeaDronesSee Object Detection v2: open https://seadronessee.cs.uni-tuebingen.de/dataset ,
# section Downloads -> "Object Detection v2", copy the download link and paste it here.
# A Nextcloud share link (cloud.cs.uni-tuebingen.de/index.php/s/XXXX) works as is.
# Compressed Version of Object Detection v2 (Nextcloud folder download, arrives as one zip).
SDS_URL = 'https://cloud.cs.uni-tuebingen.de/index.php/s/ZZxX65FGnQ8zjBP/download?path=/&files=Compressed%20Version'

# UAVDT Benchmark-M (images + annotations), Google Drive file from https://sites.google.com/view/grli-uavdt/
UAVDT_GDRIVE_ID = '1m8KA6oPIRK_Iwt9TYFquC87vBc_8wRVc'
# --------------------

RAW_DIR    = f'{DATA_DIR}/Raw'             # original archives exactly as downloaded, one copy, never edited
LOCAL      = '/content/data'               # extracted data (local disk, lost when the runtime ends)
SDS_LOCAL   = f'{LOCAL}/seadronessee'
UAVDT_LOCAL = f'{LOCAL}/uavdt'
SEED = 42

## 1. Mount Drive and check free space

In [9]:
import os, shutil, subprocess, json, random, re, glob, hashlib, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Stop if the project folder is not where we expect (wrong account or path): never create a second tree.
assert os.path.isdir(DATA_DIR), f'Not found: {DATA_DIR}. Check the Google account used to mount Drive.'
for d in (RAW_DIR, LOCAL):
    os.makedirs(d, exist_ok=True)

# Team naming rule: YYYY-MM-DD_Topic_vN.ext (00_README.txt). vN goes up if this step was already run today.
TODAY = time.strftime('%Y-%m-%d')
_done = [int(m.group(1)) for f in os.listdir(DATA_DIR)
         if (m := re.match(rf'{TODAY}_Step1Summary_v(\d+)\.json$', f))]
VERSION = max(_done, default=0) + 1
def out_name(topic, ext): return f'{TODAY}_{topic}_v{VERSION}.{ext}'
SUMMARY_PATH = f'{DATA_DIR}/{out_name("Step1Summary", "json")}'
REPORT_DIR   = f'{DATA_DIR}/{TODAY}_Step1Exploration_v{VERSION}'   # CSV tables and sample images
os.makedirs(REPORT_DIR, exist_ok=True)
print('summary will be saved as', SUMMARY_PATH)

def gb(x): return f'{x/1e9:.1f} GB'
for name, path in [('Colab local disk', '/content'), ('Google Drive', '/content/drive/MyDrive')]:
    t, u, f = shutil.disk_usage(path)
    print(f'{name:18s} free {gb(f)} of {gb(t)}')
print('\nThe archives need space twice: once on Drive (01_Data/Raw) and once extracted on the local disk.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
summary will be saved as /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/2026-10-17_Step1Summary_v1.json
Colab local disk   free 69.9 GB of 115.7 GB
Google Drive       free 66.4 GB of 115.7 GB

The archives need space twice: once on Drive (01_Data/Raw) and once extracted on the local disk.


## 2. Helpers: download (resumable) and extract

In [10]:
def sh(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'command failed ({r.returncode})')
    return r

def nextcloud_direct(url):
    # https://host/index.php/s/TOKEN  ->  https://host/index.php/s/TOKEN/download
    m = re.match(r'(https?://[^ ]+/index\.php/s/[A-Za-z0-9]+)/?$', url)
    return m.group(1) + '/download' if m else url

def download(url, out_path):
    """Resumable download with wget -c. Skips if the file is already complete."""
    url = nextcloud_direct(url)
    sh(f'wget -c --progress=dot:giga -O "{out_path}" "{url}"')
    print('saved', out_path, gb(os.path.getsize(out_path)))
    return out_path

def extract(archive, dest):
    """Extract zip / tar archives once (a marker file prevents re-extracting)."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    marker = dest / f'.extracted_{Path(archive).name}.done'
    if marker.exists():
        print('already extracted:', archive); return
    a = str(archive)
    if a.endswith('.zip'):
        sh(f'unzip -q -o "{a}" -d "{dest}"')
    elif a.endswith(('.tar', '.tar.gz', '.tgz')):
        sh(f'tar -xf "{a}" -C "{dest}"')
    else:
        raise ValueError(f'unknown archive type: {a}')
    marker.touch()

def extract_nested(dest):
    """Some releases contain zips inside the zip (UAVDT does). Extract them in place."""
    for z in sorted(Path(dest).rglob('*.zip')):
        extract(z, z.parent)

def md5(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

## 3. SeaDronesSee: download to Drive, extract locally

If the link points to several files (images and annotations separately), download each one into `01_Data/Raw/SeaDronesSee/` the same way and extract all of them.

In [11]:
sds_raw = Path(RAW_DIR) / 'SeaDronesSee'; sds_raw.mkdir(parents=True, exist_ok=True)
existing = sorted(p for p in sds_raw.iterdir() if p.suffix in ('.zip', '.tar', '.gz', '.tgz'))
if not existing:
    assert SDS_URL, 'Paste the Object Detection v2 link into SDS_URL (cell 0) first.'
    download(SDS_URL, str(sds_raw / 'SeaDronesSee_ODv2_CompressedVersion.zip'))
    existing = sorted(p for p in sds_raw.iterdir() if p.suffix in ('.zip', '.tar', '.gz', '.tgz'))
for a in existing:
    print(a.name, gb(a.stat().st_size), 'md5', md5(a))   # md5 goes into the dataset card
    extract(a, SDS_LOCAL)
extract_nested(SDS_LOCAL)
sh(f'du -sh "{SDS_LOCAL}"')

SeaDronesSee_ODv2_CompressedVersion.zip 10.0 GB md5 69b2b6d736b3cd94acd3ac4a4cb6014a
already extracted: /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/Raw/SeaDronesSee/SeaDronesSee_ODv2_CompressedVersion.zip
$ du -sh "/content/data/seadronessee"
9.4G	/content/data/seadronessee



CompletedProcess(args='du -sh "/content/data/seadronessee"', returncode=0, stdout='9.4G\t/content/data/seadronessee\n', stderr='')

## 4. UAVDT: download to Drive, extract locally

If `gdown` stops with a quota message ("Too many users have viewed or downloaded this file"), open the link below in the browser, choose **Download**, upload the zip into `01_Data/Raw/UAVDT/` on Drive, then re-run this cell.
https://drive.google.com/file/d/1m8KA6oPIRK_Iwt9TYFquC87vBc_8wRVc/view

In [12]:
!pip -q install gdown
import gdown
uav_raw = Path(RAW_DIR) / 'UAVDT'; uav_raw.mkdir(parents=True, exist_ok=True)
existing = sorted(uav_raw.glob('*.zip'))
if not existing:
    gdown.download(id=UAVDT_GDRIVE_ID, output=str(uav_raw) + '/', quiet=False)
    existing = sorted(uav_raw.glob('*.zip'))
for a in existing:
    print(a.name, gb(a.stat().st_size), 'md5', md5(a))
    extract(a, UAVDT_LOCAL)
extract_nested(UAVDT_LOCAL)
sh(f'du -sh "{UAVDT_LOCAL}"')

UAV-benchmark-M.zip 6.8 GB md5 eba1bb95b1cb7eac28cf908d41edd20e
already extracted: /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/Raw/UAVDT/UAV-benchmark-M.zip
$ du -sh "/content/data/uavdt"
6.5G	/content/data/uavdt



CompletedProcess(args='du -sh "/content/data/uavdt"', returncode=0, stdout='6.5G\t/content/data/uavdt\n', stderr='')

## 5. Exploration helpers (shared by both datasets)

In [13]:
import numpy as np, pandas as pd
from collections import Counter, defaultdict
from PIL import Image
random.seed(SEED); np.random.seed(SEED)
pd.set_option('display.width', 160)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def tree(root, depth=3, max_items=12):
    """Print the folder layout with file counts, to see how the release is organised."""
    root = Path(root)
    def walk(p, d):
        if d > depth: return
        subs = sorted([q for q in p.iterdir() if q.is_dir()])
        files = [q for q in p.iterdir() if q.is_file()]
        exts = Counter(q.suffix.lower() for q in files)
        print('  ' * d + f'{p.name}/  files={len(files)} {dict(exts.most_common(4))}')
        for q in subs[:max_items]: walk(q, d + 1)
        if len(subs) > max_items: print('  ' * (d + 1) + f'... {len(subs) - max_items} more folders')
    walk(root, 0)

def check_readable(paths, n=300):
    """Open a random sample of images fully; returns the unreadable ones.
    The full scan over every file is part of Step 2 (cleaning)."""
    paths = list(paths); sample = random.sample(paths, min(n, len(paths))); bad = []
    for p in sample:
        try:
            with Image.open(p) as im: im.load()
        except Exception as e:
            bad.append((str(p), repr(e)[:80]))
    return len(sample), bad

def box_size_table(df, by='category'):
    """Box side length in pixels: sqrt(w*h). Small objects drive the crop size choice."""
    s = np.sqrt(df.w.clip(lower=0) * df.h.clip(lower=0))
    df = df.assign(side=s)
    g = df.groupby(by).side
    out = pd.DataFrame({'boxes': g.size(), 'median_px': g.median().round(1),
                        'p10_px': g.quantile(.1).round(1), 'p90_px': g.quantile(.9).round(1),
                        'share_<16px': g.apply(lambda x: (x < 16).mean()).round(3),
                        'share_<32px': g.apply(lambda x: (x < 32).mean()).round(3)})
    return out.sort_values('boxes', ascending=False)

SUMMARY = {'seed': SEED, 'created': time.strftime('%Y-%m-%d %H:%M')}

## 6. SeaDronesSee: structure, annotations and metadata

The key question for the split: does every image carry a **flight / video id** (`source.video` or `source.folder_name`)? If yes, Step 3 groups by it. If not, Step 3 groups by capture time and GPS instead.

In [14]:
tree(SDS_LOCAL)
sds_imgs_on_disk = {p.name: p for p in Path(SDS_LOCAL).rglob('*') if p.suffix.lower() in IMG_EXT}
print('\nimage files on disk:', len(sds_imgs_on_disk))
sds_jsons = sorted(p for p in Path(SDS_LOCAL).rglob('*.json') if not p.name.startswith('.'))
print('json files:'); [print('  ', p.relative_to(SDS_LOCAL), gb(p.stat().st_size)) for p in sds_jsons]

seadronessee/  files=1 {'.done': 1}
  Compressed Version/  files=0 {}
    annotations/  files=2 {'.json': 2}
    images/  files=0 {}
      test/  files=3750 {'.jpg': 3750}
      train/  files=8930 {'.jpg': 8930}
      val/  files=1547 {'.jpg': 1547}

image files on disk: 14227
json files:
   Compressed Version/annotations/instances_train.json 0.0 GB
   Compressed Version/annotations/instances_val.json 0.0 GB


[None, None]

In [15]:
def split_name(p):
    n = p.stem.lower()
    for s in ('train', 'val', 'test'):
        if s in n: return s
    return n

sds_img_rows, sds_box_rows, sds_split_info = [], [], {}
for jp in sds_jsons:
    with open(jp) as f: J = json.load(f)
    if not isinstance(J, dict) or 'images' not in J:
        print('skip (not COCO):', jp.name); continue
    split = split_name(jp)
    cats = {c['id']: c['name'] for c in J.get('categories', [])}
    anns = J.get('annotations', [])
    sds_split_info[split] = {'file': jp.name, 'images': len(J['images']), 'annotations': len(anns), 'categories': cats}
    print(f"\n[{split}] {jp.name}: {len(J['images'])} images, {len(anns)} boxes, categories={cats}")
    print('  keys of one image entry:', sorted(J['images'][0].keys()))
    if 'source' in J['images'][0]: print('  example source:', J['images'][0]['source'])
    if 'meta' in J['images'][0]:   print('  example meta keys:', sorted((J['images'][0]['meta'] or {}).keys()))
    for im in J['images']:
        src = im.get('source') or {}; meta = im.get('meta') or {}
        sds_img_rows.append({'split': split, 'image_id': im['id'], 'file_name': im['file_name'],
            'width': im.get('width'), 'height': im.get('height'),
            'video': src.get('video') or src.get('folder_name'), 'drone': src.get('drone'),
            'frame_no': src.get('frame_no'), 'altitude': meta.get('altitude'),
            'gimbal_pitch': meta.get('gimbal_pitch'), 'date_time': meta.get('date_time') or im.get('date_time'),
            'on_disk': Path(im['file_name']).name in sds_imgs_on_disk})
    for a in anns:
        x, y, w, h = a['bbox']
        sds_box_rows.append({'split': split, 'image_id': a['image_id'],
            'category': cats.get(a['category_id'], a['category_id']), 'x': x, 'y': y, 'w': w, 'h': h})

sds_img = pd.DataFrame(sds_img_rows); sds_box = pd.DataFrame(sds_box_rows)
print('\nimages per split:\n', sds_img.groupby('split').size())


[train] instances_train.json: 8930 images, 57760 boxes, categories={0: 'ignored', 1: 'swimmer', 2: 'boat', 3: 'jetski', 4: 'life_saving_appliances', 5: 'buoy'}
  keys of one image entry: ['date_time', 'file_name', 'height', 'id', 'meta', 'source', 'width']
  example source: {'drone': 'mavic', 'folder_name': 'DJI_0032', 'video': 'DJI_0032.MP4', 'frame_no': 0}
  example meta keys: ['compass_heading(degrees)', 'datetime(utc)', 'gimbal_heading(degrees)', 'gimbal_pitch(degrees)', 'height_above_takeoff(meter)', 'image_name', 'latitude', 'longitude', 'speed(m/s)', 'xspeed(m/s)', 'yspeed(m/s)', 'zspeed(m/s)']

[val] instances_val.json: 1547 images, 9630 boxes, categories={0: 'ignored', 1: 'swimmer', 2: 'boat', 3: 'jetski', 4: 'life_saving_appliances', 5: 'buoy'}
  keys of one image entry: ['date_time', 'file_name', 'height', 'id', 'meta', 'source', 'width']
  example source: {'drone': 'mavic', 'folder_name': 'DJI_0032', 'video': 'DJI_0032.MP4', 'frame_no': 5415}
  example meta keys: ['compass

In [16]:
# Class balance per split (boxes = future classification crops)
print('boxes per class and split:')
cls_split = sds_box.pivot_table(index='category', columns='split', values='w', aggfunc='size', fill_value=0)
display(cls_split)

print('\nimage resolutions:'); display(sds_img.groupby(['width', 'height']).size().rename('images'))
print('\nbox sizes (pixels):'); display(box_size_table(sds_box))

# Metadata needed for the leakage-safe split
m = sds_img.assign(has_video=sds_img.video.notna(), has_alt=sds_img.altitude.notna())
print('\nshare of images with a flight/video id and with altitude, per split:')
display(m.groupby('split')[['has_video', 'has_alt', 'on_disk']].mean().round(3))
vids = sds_img.dropna(subset=['video'])
per_video = vids.groupby('video').agg(images=('image_id', 'size'), splits=('split', lambda s: ','.join(sorted(set(s)))))
print(f'\ndistinct flights/videos: {len(per_video)}')
print('images per flight/video (describe):'); display(per_video.images.describe().round(1))
print('flights/videos that appear in more than one official split (would leak if kept as is):',
      int((per_video.splits.str.contains(',')).sum()))
print('\naltitude (m):'); display(sds_img.altitude.describe().round(1))

boxes per class and split:


split,train,val
category,,
boat,13022,2214
buoy,4389,560
jetski,2330,320
life_saving_appliances,923,330
swimmer,37096,6206



image resolutions:


width  height
1225   926          1
1227   931          1
1228   933          1
       934          1
1229   932          5
       933          3
       934          7
       935          1
1230   932         28
       933        139
       934         42
       935          2
1231   932          7
       933        124
       934         91
1232   933          3
       934         16
       935          1
1233   933          1
       935          1
1920   1080      1834
3632   5456       329
3840   2160      6830
5456   3632      1009
Name: images, dtype: int64


box sizes (pixels):


...

,has_video,has_alt,on_disk
split,,,
train,1.0,0.046,1.0
val,1.0,0.041,1.0



distinct flights/videos: 39
images per flight/video (describe):


,images
count,39.0
mean,268.6
std,294.2
min,1.0
25%,53.5
50%,219.0
75%,403.5
max,1338.0


flights/videos that appear in more than one official split (would leak if kept as is): 34

altitude (m):


,altitude
count,475.0
mean,134.3
std,71.3
min,52.6
25%,79.5
50%,110.5
75%,226.6
max,259.4


In [17]:
n_checked, bad = check_readable(sds_imgs_on_disk.values(), n=300)
print(f'readable check: {n_checked - len(bad)}/{n_checked} OK'); [print('  BAD', b) for b in bad[:10]]

SUMMARY['seadronessee'] = {
    'splits': sds_split_info,
    'image_files_on_disk': len(sds_imgs_on_disk),
    'boxes_per_class_split': cls_split.to_dict(),
    'resolutions': {f'{w}x{h}': int(n) for (w, h), n in sds_img.groupby(['width', 'height']).size().items()},
    'share_with_video_id': round(float(sds_img.video.notna().mean()), 3),
    'distinct_videos': int(len(per_video)),
    'videos_in_several_splits': int(per_video.splits.str.contains(',').sum()),
    'altitude_m': sds_img.altitude.describe().round(1).to_dict(),
    'box_size_px': box_size_table(sds_box).to_dict(orient='index'),
    'readable_sample': [n_checked, len(bad)],
}
sds_img.to_csv(f'{REPORT_DIR}/{out_name("SDSImages", "csv")}', index=False)
per_video.to_csv(f'{REPORT_DIR}/{out_name("SDSPerVideo", "csv")}')

readable check: 300/300 OK


## 7. UAVDT: sequences, annotations and class balance

Annotation columns of `*_gt_whole.txt` (UAVDT toolkit): `frame_index, target_id, left, top, width, height, out_of_view, occlusion, object_category` with category 1 = car, 2 = truck, 3 = bus. The split unit is the **sequence** (M0101, M0202, ...).

In [18]:
tree(UAVDT_LOCAL, depth=2)
seq_dirs = {p.name: p for p in Path(UAVDT_LOCAL).rglob('M[0-9][0-9][0-9][0-9]') if p.is_dir()}
gt_files = {re.match(r'(M\d{4})', p.name).group(1): p for p in Path(UAVDT_LOCAL).rglob('M*_gt_whole.txt')}
print(f'\nsequence image folders: {len(seq_dirs)}   gt_whole files: {len(gt_files)}')
print('sequences with images but no GT:', sorted(set(seq_dirs) - set(gt_files))[:20])
print('sequences with GT but no images:', sorted(set(gt_files) - set(seq_dirs))[:20])

uavdt/  files=1 {'.done': 1}
  UAV-benchmark-M/  files=0 {}
    M0101/  files=407 {'.jpg': 407}
    M0201/  files=1076 {'.jpg': 1076}
    M0202/  files=291 {'.jpg': 291}
    M0203/  files=1007 {'.jpg': 1007}
    M0204/  files=350 {'.jpg': 350}
    M0205/  files=646 {'.jpg': 646}
    M0206/  files=562 {'.jpg': 562}
    M0207/  files=885 {'.jpg': 885}
    M0208/  files=265 {'.jpg': 265}
    M0209/  files=1576 {'.jpg': 1576}
    M0210/  files=583 {'.jpg': 583}
    M0301/  files=325 {'.jpg': 325}
    ... 38 more folders

sequence image folders: 50   gt_whole files: 0
sequences with images but no GT: ['M0101', 'M0201', 'M0202', 'M0203', 'M0204', 'M0205', 'M0206', 'M0207', 'M0208', 'M0209', 'M0210', 'M0301', 'M0401', 'M0402', 'M0403', 'M0501', 'M0601', 'M0602', 'M0603', 'M0604']
sequences with GT but no images: []


In [20]:
import zipfile
from pathlib import Path
RAW_U = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/Raw/UAVDT'
DST = Path('/content/data/UAVDT')
for z in ['UAV-benchmark-MOTD_v1.0.zip', 'M_attr.zip']:
    zipfile.ZipFile(f'{RAW_U}/{z}').extractall(DST)
gt_files = {p.name.split('_')[0]: p for p in sorted(DST.rglob('*_gt_whole.txt'))}
print(len(gt_files), 'GT sequences | image sequences found:', len(seq_dirs))

UAV_CLASSES = {1: 'car', 2: 'truck', 3: 'bus'}
COLS = ['frame', 'target_id', 'left', 'top', 'w', 'h', 'out_of_view', 'occlusion', 'cat']
seq_rows, uav_parts = [], []
for seq in sorted(gt_files):
    g = pd.read_csv(gt_files[seq], header=None, names=COLS)
    g['seq'] = seq; g['category'] = g.cat.map(UAV_CLASSES).fillna(g.cat.astype(str))
    uav_parts.append(g)
    frames = sorted(seq_dirs[seq].glob('*.jpg')) if seq in seq_dirs else []
    first = None
    if frames:
        with Image.open(frames[0]) as im: first = im.size
    seq_rows.append({'seq': seq, 'frames_on_disk': len(frames), 'frames_annotated': g.frame.nunique(),
                     'boxes': len(g), 'tracks': g.target_id.nunique(),
                     'car': int((g.cat == 1).sum()), 'truck': int((g.cat == 2).sum()), 'bus': int((g.cat == 3).sum()),
                     'resolution': f'{first[0]}x{first[1]}' if first else None,
                     'frames_at_5fps': len(frames) // 6})   # assumes 30 fps video: keep 1 frame in 6
uav_box = pd.concat(uav_parts, ignore_index=True)
uav_seq = pd.DataFrame(seq_rows).set_index('seq')
print('category values found in GT:', sorted(int(v) for v in uav_box.cat.unique()))
display(uav_seq.describe().round(1))
print('\nboxes per class (all sequences):'); display(uav_box.category.value_counts())
print('\ndistinct physical objects (tracks) per class; crops of one track are near-duplicates:')
display(uav_box.drop_duplicates(['seq', 'target_id']).category.value_counts().rename('tracks'))
print('\nsequences that contain each class:'); display((uav_seq[['car', 'truck', 'bus']] > 0).sum())
print('\nresolutions:'); display(uav_seq.resolution.value_counts())
print('\nbox sizes (pixels):'); display(box_size_table(uav_box))
print('\nout_of_view levels:', uav_box.out_of_view.value_counts().to_dict(), ' occlusion levels:', uav_box.occlusion.value_counts().to_dict())

50 GT sequences | image sequences found: 50
category values found in GT: [1, 2, 3]


,frames_on_disk,frames_annotated,boxes,tracks,car,truck,bus,frames_at_5fps
count,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
mean,814.7,808.2,15975.9,53.1,15113.8,501.7,360.4,135.4
std,451.4,452.5,15875.4,33.3,15169.3,785.9,736.7,75.2
min,265.0,265.0,790.0,3.0,491.0,0.0,0.0,44.0
25%,474.0,474.0,5425.0,27.0,5144.8,0.0,0.0,78.5
50%,671.5,659.5,11283.0,49.5,10947.5,201.5,2.5,111.5
75%,1078.2,1078.2,21371.2,69.0,19694.8,637.2,354.2,179.0
max,2035.0,2035.0,94043.0,182.0,90563.0,4188.0,3699.0,339.0



boxes per class (all sequences):


,count
category,
car,755688
truck,25086
bus,18021



distinct physical objects (tracks) per class; crops of one track are near-duplicates:


...


resolutions:


,count
resolution,
1024x540,49
960x540,1



box sizes (pixels):


,boxes,median_px,p10_px,p90_px,share_<16px,share_<32px
category,,,,,,
car,755688,27.6,14.5,47.7,0.150,0.636
truck,25086,37.9,17.9,66.9,0.056,0.349
bus,18021,50.8,20.6,94.7,0.046,0.233



out_of_view levels: {1: 725059, 2: 37089, 3: 36647}  occlusion levels: {1: 715470, 4: 64004, 3: 10025, 2: 9296}


In [21]:
all_uav_frames = [p for d in seq_dirs.values() for p in d.glob('*.jpg')]
n_checked, bad = check_readable(all_uav_frames, n=300)
print(f'readable check: {n_checked - len(bad)}/{n_checked} OK'); [print('  BAD', b) for b in bad[:10]]

SUMMARY['uavdt'] = {
    'sequences_with_images': len(seq_dirs), 'sequences_with_gt': len(gt_files),
    'frames_on_disk': int(uav_seq.frames_on_disk.sum()), 'boxes': int(len(uav_box)),
    'boxes_per_class': uav_box.category.value_counts().to_dict(),
    'sequences_per_class': (uav_seq[['car', 'truck', 'bus']] > 0).sum().to_dict(),
    'resolutions': uav_seq.resolution.value_counts().to_dict(),
    'frames_at_5fps_total': int(uav_seq.frames_at_5fps.sum()),
    'box_size_px': box_size_table(uav_box).to_dict(orient='index'),
    'readable_sample': [n_checked, len(bad)],
}
uav_seq.to_csv(f'{REPORT_DIR}/{out_name("UAVDTPerSequence", "csv")}')

readable check: 300/300 OK


## 8. Look at the data: random images with their boxes

In [22]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4', '#f032e6', '#bfef45']

def show(img_path, boxes, title, out_png):
    """boxes: list of (x, y, w, h, label)."""
    with Image.open(img_path) as im: im = im.convert('RGB')
    fig, ax = plt.subplots(figsize=(12, 7)); ax.imshow(im); ax.set_title(title); ax.axis('off')
    labels = sorted({b[4] for b in boxes})
    for x, y, w, h, lab in boxes:
        c = COLORS[labels.index(lab) % len(COLORS)]
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=1.5, color=c))
        ax.text(x, y - 4, lab, color=c, fontsize=8)
    fig.savefig(out_png, dpi=90, bbox_inches='tight'); plt.show(); plt.close(fig)

# SeaDronesSee: 3 random annotated images
groups = list(sds_box.groupby(['split', 'image_id']))
fn_of = sds_img.set_index(['split', 'image_id']).file_name
for i, (key, grp) in enumerate(random.sample(groups, min(3, len(groups)))):
    fn = Path(fn_of[key]).name
    if fn in sds_imgs_on_disk:
        show(sds_imgs_on_disk[fn], [(r.x, r.y, r.w, r.h, r.category) for r in grp.itertuples()],
             f'SeaDronesSee [{key[0]}] {fn}', f'{REPORT_DIR}/{out_name(f"SampleSDS{i}", "png")}')

# UAVDT: one random annotated frame from 3 random sequences
for i, seq in enumerate(random.sample(sorted(set(gt_files) & set(seq_dirs)), min(3, len(set(gt_files) & set(seq_dirs))))):
    g = uav_box[uav_box.seq == seq]; fr = int(random.choice(g.frame.unique()))
    img = seq_dirs[seq] / f'img{fr:06d}.jpg'
    if img.exists():
        show(img, [(r.left, r.top, r.w, r.h, r.category) for r in g[g.frame == fr].itertuples()],
             f'UAVDT {seq} frame {fr}', f'{REPORT_DIR}/{out_name(f"SampleUAVDT{i}", "png")}')

Output hidden; open in https://colab.research.google.com to view.

## 9. Save the summary

Send me the printed summary (or the file `01_Data/YYYY-MM-DD_Step1Summary_vN.json`). It decides Step 2 (cleaning) and Step 3 (split): which id to group by, which classes are rare, and the crop size.

In [23]:
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, float) and o != o: return None
    return o
out = SUMMARY_PATH
SUMMARY['version'] = VERSION; SUMMARY['report_dir'] = REPORT_DIR
with open(out, 'w') as f: json.dump(jsonable(SUMMARY), f, indent=2, ensure_ascii=False)
print('saved', out, '\n'); print(json.dumps(jsonable(SUMMARY), indent=1, ensure_ascii=False)[:6000])

saved /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/2026-10-17_Step1Summary_v1.json 

{
 "seed": 42,
 "created": "2026-10-17 15:10",
 "seadronessee": {
  "splits": {
   "train": {
    "file": "instances_train.json",
    "images": 8930,
    "annotations": 57760,
    "categories": {
     "0": "ignored",
     "1": "swimmer",
     "2": "boat",
     "3": "jetski",
     "4": "life_saving_appliances",
     "5": "buoy"
    }
   },
   "val": {
    "file": "instances_val.json",
    "images": 1547,
    "annotations": 9630,
    "categories": {
     "0": "ignored",
     "1": "swimmer",
     "2": "boat",
     "3": "jetski",
     "4": "life_saving_appliances",
     "5": "buoy"
    }
   }
  },
  "image_files_on_disk": 14227,
  "boxes_per_class_split": {
   "train": {
    "boat": 13022,
    "buoy": 4389,
    "jetski": 2330,
    "life_saving_appliances": 923,
    "swimmer": 37096
   },
   "val": {
    "boat": 2214,
    "buoy": 560,
    "jetski": 320,
    "life_saving_appliances": 330,